# Parte 2 — Dados, pré-processamento e Inception-v4 com base congelada

**Projeto de CNN · Machine Learning e Deep Learning (CC6NA 2026.2) · Profa. Polyana Nascimento**
Equipe Inception (vaga 1): Alberto Acosta, Davi Corrêa, Gabriel Albuquerque

Este notebook cobre as **etapas 2 e 3 da lauda**:

| Etapa | Conteúdo |
|---|---|
| 2 | Análise inicial das imagens, limpeza, divisão estratificada treino/validação/teste, pré-processamento e métricas |
| 3 | Inception-v4 pré-treinada (ImageNet), classificador novo, **base congelada**, curvas, resultados na validação e modelo salvo |

Ele gera os arquivos que a parte 3 (fine-tuning A e B) usa como ponto de partida:

- `artifacts/split.csv`: a divisão fixa dos dados, a mesma para as três condições;
- `artifacts/modelo_base_congelada.pt`: o modelo de referência. As estratégias A e B partem de cópias dele;
- `artifacts/val_preds_base_congelada.csv` e `artifacts/resumo_base_congelada.json`: previsões e métricas para a comparação.

**Ambiente:** Google Colab ou Kaggle com GPU (T4 é suficiente). Dependências: `torch`, `torchvision`, `timm`, `kagglehub`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`, `Pillow`.

## 0. Ambiente e configuração

In [ ]:
# No Colab, torch/torchvision/sklearn já vêm instalados; timm e kagglehub às vezes não.
%pip install -q timm kagglehub

In [ ]:
import os, json, time, copy, random, hashlib, platform
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms as T
import timm
from timm.data import resolve_data_config

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

ImageFile.LOAD_TRUNCATED_IMAGES = False  # imagem truncada deve falhar na verificação, não passar escondida
sns.set_theme(style="whitegrid")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__)
print("Dispositivo:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

In [ ]:
# ---------------- Configuração (tudo que define o experimento fica aqui) ----------------
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"   # só para testar o código rapidamente em CPU

SEED = 42
MODEL_NAME = "inception_v4.tf_in1k"   # Inception-v4 com pesos ImageNet-1k (portados do TF-Slim oficial)
PRETRAINED = not SMOKE_TEST

TEST_SIZE = 0.15                       # 70% treino / 15% validação / 15% teste
VAL_SIZE = 0.15
MIN_IMGS_PER_CLASS = 10 if SMOKE_TEST else 100   # exigência da lauda: >= 100 imagens por classe

# Classes retiradas do problema (a lauda validada fala em 19 classes; o dataset original tem 23).
# Preencher com as classes que a equipe decidiu remover — ver discussão na seção 2.4.
CLASSES_EXCLUIR = []

BATCH_SIZE = 8 if SMOKE_TEST else 32
NUM_WORKERS = 0 if SMOKE_TEST else 2
LR = 1e-3                 # só o classificador treina, então lr "alto" é seguro
WEIGHT_DECAY = 1e-4
DROP_RATE = 0.3           # dropout antes do classificador novo
MAX_EPOCHS = 2 if SMOKE_TEST else 20
PATIENCE = 4              # early stopping: épocas sem melhorar o F1-macro da validação
MONITOR = "val_f1_macro"  # regra de seleção — deve ser a MESMA nas três condições

ARTIFACTS = Path(os.environ.get("ARTIFACTS_DIR", "artifacts"))
ARTIFACTS.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()

In [ ]:
# (Opcional, Colab) salvar os artefatos no Drive, para não perder o modelo quando a sessão cair.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = Path("/content/drive/MyDrive/cnn-inceptionv4-sea-animals/artifacts")
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("Artefatos em:", ARTIFACTS.resolve())

## 1. Dataset

**Sea Animals Image Dataset 🌊** — Kaggle, `vencerlanz09/sea-animals-image-dataste`
<https://www.kaggle.com/datasets/vencerlanz09/sea-animals-image-dataste>

Fotos de criaturas marinhas organizadas em uma pasta por classe (o rótulo é o nome da pasta). As imagens vêm de fontes variadas da internet: tamanhos, proporções, formatos e qualidade diferentes, algumas com fundo de aquário, outras subaquáticas, outras fora d'água. Isso é exatamente o que a análise abaixo precisa examinar.

O download usa `kagglehub` (no Colab pode pedir o login do Kaggle). Se o dataset já estiver em disco, defina a variável de ambiente `SEA_DATA_DIR` com o caminho.

In [ ]:
DATA_DIR = os.environ.get("SEA_DATA_DIR")
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("vencerlanz09/sea-animals-image-dataste")
DATA_DIR = Path(DATA_DIR)
print("Dataset em:", DATA_DIR)
print("Conteúdo da raiz:", sorted(p.name for p in DATA_DIR.iterdir())[:30])

## 2. Análise inicial e pré-processamento (etapa 2)

### 2.1 Inventário e verificação de cada arquivo

Para cada imagem: abre, decodifica por completo, registra largura, altura, modo de cor (RGB, RGBA, P, L…), formato real e o hash MD5 do arquivo. O hash serve para achar **duplicatas exatas**, que se caíssem uma no treino e outra no teste seriam **vazamento de dados**.

In [ ]:
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tif", ".tiff"}

files = [p for p in sorted(DATA_DIR.rglob("*")) if p.is_file() and p.suffix.lower() in EXTS]
df = pd.DataFrame({
    "path": [str(p.relative_to(DATA_DIR)) for p in files],   # caminho relativo: funciona em qualquer máquina
    "label": [p.parent.name for p in files],
    "ext": [p.suffix.lower() for p in files],
})
outros = [p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() not in EXTS]
print(f"{len(df)} imagens em {df.label.nunique()} pastas | {len(outros)} arquivos ignorados (não-imagem)")

def inspect_image(rel_path):
    info = {"path": rel_path, "erro": ""}
    full = DATA_DIR / rel_path
    try:
        data = full.read_bytes()
        info["md5"] = hashlib.md5(data).hexdigest()
        with Image.open(full) as im:
            info.update(width=im.width, height=im.height, mode=im.mode, fmt=im.format,
                        n_frames=getattr(im, "n_frames", 1))
            im.load()  # decodifica de verdade: pega arquivo truncado/corrompido
        info["ok"] = True
    except Exception as e:
        info["ok"] = False
        info["erro"] = repr(e)[:150]
    return info

t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    info = pd.DataFrame(list(ex.map(inspect_image, df.path)))
df = df.merge(info, on="path")
print(f"Verificação em {time.time() - t0:.0f}s")
df.head()

In [ ]:
print("Arquivos ilegíveis:", (~df.ok).sum())
display(df.loc[~df.ok, ["path", "erro"]].head(20))

print("\nModo de cor:"); print(df.loc[df.ok, "mode"].value_counts().to_string())
print("\nFormato real (pelo conteúdo, não pela extensão):"); print(df.loc[df.ok, "fmt"].value_counts().to_string())
print("\nExtensão que não bate com o formato real:",
      int(((df.ext.isin([".jpg", ".jpeg"])) & (df.fmt != "JPEG") & df.ok).sum()))
print("Imagens animadas (mais de 1 quadro):", int((df.n_frames.fillna(1) > 1).sum()))

### 2.2 Distribuição das classes

In [ ]:
counts = df[df.ok].label.value_counts()
fig, ax = plt.subplots(figsize=(11, 4))
counts.plot.bar(ax=ax, color="#4C72B0")
ax.axhline(MIN_IMGS_PER_CLASS, color="crimson", ls="--", label=f"mínimo da lauda ({MIN_IMGS_PER_CLASS})")
ax.set_ylabel("imagens"); ax.set_title("Imagens por classe (antes da limpeza)"); ax.legend()
plt.tight_layout(); plt.show()

print(f"Maior classe: {counts.idxmax()} ({counts.max()}) | menor: {counts.idxmin()} ({counts.min()}) "
      f"| razão de desbalanceamento: {counts.max() / counts.min():.1f}x")
counts.to_frame("imagens")

### 2.3 Tamanho e proporção das imagens

A Inception-v4 recebe entradas de **299×299**. É preciso saber quanto as imagens originais diferem disso: imagens muito pequenas serão ampliadas (perdem nitidez) e proporções muito alongadas sofrem com o recorte central.

In [ ]:
ok = df[df.ok].copy()
ok["aspect"] = ok.width / ok.height
ok["min_side"] = ok[["width", "height"]].min(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(ok.width, ok.height, s=3, alpha=.3); axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].axvline(299, c="crimson", ls="--"); axes[0].axhline(299, c="crimson", ls="--")
axes[0].set_xlabel("largura (px)"); axes[0].set_ylabel("altura (px)"); axes[0].set_title("Dimensões (linhas = 299 px)")
axes[1].hist(ok.aspect.clip(0, 3), bins=60); axes[1].set_title("Proporção largura/altura"); axes[1].set_xlabel("largura / altura")
axes[2].hist(ok.min_side.clip(0, 1500), bins=60); axes[2].axvline(299, c="crimson", ls="--")
axes[2].set_title("Menor lado (px)"); axes[2].set_xlabel("px")
plt.tight_layout(); plt.show()

print(ok[["width", "height", "aspect", "min_side"]].describe().round(2).to_string())
print(f"\nImagens com menor lado < 299 px: {(ok.min_side < 299).mean():.1%}")
print(f"Imagens muito alongadas (proporção > 2 ou < 0,5): {((ok.aspect > 2) | (ok.aspect < .5)).mean():.1%}")

In [ ]:
def load_rgb(path):
    # Converte qualquer modo para RGB. Transparência (RGBA, LA, P com transparência) vai sobre fundo branco,
    # em vez de virar preto, que é o que um convert("RGB") direto faria.
    with Image.open(path) as im:
        if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
            im = im.convert("RGBA")
            bg = Image.new("RGB", im.size, (255, 255, 255))
            bg.paste(im, mask=im.getchannel("A"))
            return bg
        return im.convert("RGB")

def show_grid(paths, titles, ncols=6, size=2.3, suptitle=None):
    n = len(paths)
    nrows = max(1, int(np.ceil(n / ncols)))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size * 1.1))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, p, t in zip(np.atleast_1d(axes).ravel(), paths, titles):
        ax.imshow(load_rgb(DATA_DIR / p)); ax.set_title(t, fontsize=8)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout(); plt.show()

amostra = ok.groupby("label").sample(n=1, random_state=SEED)
show_grid(amostra.path.tolist(), amostra.label.tolist(), suptitle="Um exemplo por classe")

### 2.4 Duplicatas, rótulos conflitantes e escolha das classes

- **Duplicata na mesma classe:** mantém uma cópia e descarta as outras (senão uma pode cair no treino e a outra no teste).
- **Mesmo arquivo em classes diferentes:** o rótulo é ambíguo, descarta todas as cópias.
- **Duplicatas quase idênticas** (mesma foto redimensionada ou recomprimida) não são pegas pelo MD5. Essa é uma **limitação** registrada: o hash só detecta cópias byte a byte.

Depois disso, removemos `CLASSES_EXCLUIR` e qualquer classe que fique abaixo do mínimo de imagens.

> **A decidir com a equipe:** a planilha validada diz **19 classes** e "foco em caranguejos (espécie de caranguejo)". O dataset tem **uma** classe *Crabs* e não separa espécies, então o alvo real é **o tipo de animal marinho**. Confirmar quais 4 classes saem para chegar a 19 (e por quê) e preencher `CLASSES_EXCLUIR` na seção 0.

In [ ]:
dup = ok.groupby("md5").agg(n=("path", "size"), n_labels=("label", "nunique"))
md5_conflito = set(dup[dup.n_labels > 1].index)
print(f"Grupos de arquivos idênticos: {(dup.n > 1).sum()} "
      f"(cópias extras: {int((dup.n - 1).clip(lower=0).sum())})")
print(f"Arquivos idênticos com rótulos diferentes: {int(ok.md5.isin(md5_conflito).sum())} imagens em {len(md5_conflito)} grupos")

if md5_conflito:
    ex = ok[ok.md5.isin(list(md5_conflito)[:4])].sort_values("md5")
    show_grid(ex.path.tolist(), ex.label.tolist(), suptitle="Mesmo arquivo em classes diferentes")

clean = ok[~ok.md5.isin(md5_conflito)].drop_duplicates("md5", keep="first")
clean = clean[~clean.label.isin(CLASSES_EXCLUIR)]
counts_clean = clean.label.value_counts()
pequenas = counts_clean[counts_clean < MIN_IMGS_PER_CLASS].index.tolist()
clean = clean[~clean.label.isin(pequenas)].reset_index(drop=True)

resumo_limpeza = pd.Series({
    "imagens encontradas": len(df),
    "ilegíveis": int((~df.ok).sum()),
    "rótulo conflitante": int(ok.md5.isin(md5_conflito).sum()),
    "duplicatas removidas": len(ok[~ok.md5.isin(md5_conflito)]) - len(ok[~ok.md5.isin(md5_conflito)].drop_duplicates("md5")),
    "classes excluídas (decisão)": len(CLASSES_EXCLUIR),
    f"classes < {MIN_IMGS_PER_CLASS} imagens": len(pequenas),
    "imagens finais": len(clean),
    "classes finais": clean.label.nunique(),
})
print(resumo_limpeza.to_string())
if pequenas:
    print("Classes removidas por terem poucas imagens:", pequenas)

### 2.5 Divisão treino / validação / teste (holdout estratificado)

- **70 / 15 / 15**, estratificado por classe, semente fixa.
- O **teste é separado primeiro** e só volta a ser usado na etapa 7, depois que o modelo recomendado for escolhido pela validação.
- A divisão é salva em `artifacts/split.csv` com caminhos relativos. **A parte 3 lê esse arquivo**, o que garante os mesmos conjuntos nas três condições.

In [ ]:
classes = sorted(clean.label.unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)

trval, test = train_test_split(clean, test_size=TEST_SIZE, stratify=clean.label, random_state=SEED)
train, val = train_test_split(trval, test_size=VAL_SIZE / (1 - TEST_SIZE), stratify=trval.label, random_state=SEED)

split = pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])
split["y"] = split.label.map(class_to_idx)
split = split[["path", "label", "y", "split", "md5", "width", "height"]].reset_index(drop=True)

# Garantias contra vazamento
assert not set(train.md5) & set(test.md5) and not set(val.md5) & set(test.md5) and not set(train.md5) & set(val.md5)
assert split.path.is_unique

split.to_csv(ARTIFACTS / "split.csv", index=False)
json.dump({"classes": classes, "class_to_idx": class_to_idx}, open(ARTIFACTS / "classes.json", "w"), indent=2, ensure_ascii=False)

tab = split.pivot_table(index="label", columns="split", values="path", aggfunc="count")[["train", "val", "test"]]
tab["total"] = tab.sum(axis=1)
tab.loc["TOTAL"] = tab.sum()
print(f"{NUM_CLASSES} classes | treino {len(train)} | validação {len(val)} | teste {len(test)}")
tab

### 2.6 Pré-processamento esperado pelos pesos pré-treinados

Os pesos `inception_v4.tf_in1k` foram treinados no ImageNet com o pré-processamento do TensorFlow-Slim (estilo Inception), diferente do usado pela ResNet ou VGG. Usar outro faz a rede ver entradas fora da distribuição em que aprendeu. Lendo a configuração do próprio modelo:

In [ ]:
_tmp = timm.create_model(MODEL_NAME, pretrained=False)
DATA_CFG = resolve_data_config({}, model=_tmp)
del _tmp
print(DATA_CFG)

IMG_SIZE = DATA_CFG["input_size"][-1]                        # 299
MEAN, STD = DATA_CFG["mean"], DATA_CFG["std"]                # (0.5, 0.5, 0.5): pixel [0,1] -> [-1,1]
RESIZE = int(round(IMG_SIZE / DATA_CFG["crop_pct"]))         # 299 / 0.875 = 342
INTERP = T.InterpolationMode.BICUBIC

| Decisão | Escolha | Justificativa |
|---|---|---|
| Tamanho de entrada | **299×299** | Resolução nativa da Inception-v4; o stem foi projetado para ela |
| Canais | **RGB, 3 canais** | Pesos ImageNet são RGB. Imagens em cinza (L) são replicadas em 3 canais; transparência vai sobre fundo branco |
| Normalização | `(x − 0,5) / 0,5` → **[−1, 1]** | É o pré-processamento estilo Inception dos pesos originais (não a média/desvio do ImageNet da ResNet) |
| Avaliação (val/teste/imagem externa) | Redimensiona o menor lado para 342 (bicúbica) e recorta o centro 299×299 | Igual ao protocolo de avaliação dos pesos (`crop_pct = 0,875`); determinístico |
| Aumento de dados (só treino) | Recorte aleatório redimensionado (escala 0,6–1,0), espelhamento horizontal, leve variação de brilho/contraste/saturação | Imagens vêm de fontes e enquadramentos variados. **Sem** espelhamento vertical nem rotação forte: animais costumam aparecer "em pé" e isso criaria imagens irreais |

In [ ]:
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), interpolation=INTERP),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([
    T.Resize(RESIZE, interpolation=INTERP),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

class SeaAnimals(Dataset):
    def __init__(self, frame, root, transform):
        self.paths = frame.path.tolist(); self.ys = frame.y.tolist()
        self.root, self.transform = Path(root), transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.transform(load_rgb(self.root / self.paths[i])), self.ys[i]

def make_loaders(split_df, batch_size=BATCH_SIZE):
    g = torch.Generator().manual_seed(SEED)
    mk = lambda name, tf, shuffle: DataLoader(
        SeaAnimals(split_df[split_df.split == name], DATA_DIR, tf), batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", generator=g if shuffle else None)
    return mk("train", train_tf, True), mk("val", eval_tf, False), mk("test", eval_tf, False)

train_loader, val_loader, test_loader = make_loaders(split)

# Visualizar o que a rede realmente recebe (desfazendo a normalização)
def denorm(x):
    return (x * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1).permute(1, 2, 0)

p0 = split[split.split == "train"].iloc[0]
ds_vis = SeaAnimals(split[split.split == "train"].iloc[:1], DATA_DIR, train_tf)
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
axes[0].imshow(load_rgb(DATA_DIR / p0.path)); axes[0].set_title(f"original ({p0.label})", fontsize=9)
axes[1].imshow(denorm(eval_tf(load_rgb(DATA_DIR / p0.path)))); axes[1].set_title("avaliação 299×299", fontsize=9)
for k, ax in enumerate(axes[2:]):
    ax.imshow(denorm(ds_vis[0][0])); ax.set_title(f"aumento #{k + 1}", fontsize=9)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

xb, yb = next(iter(train_loader))
print("Lote de treino:", tuple(xb.shape), xb.dtype, f"| faixa [{xb.min():.2f}, {xb.max():.2f}]")

### 2.7 Métricas e erros mais prejudiciais

O problema é **classificação multiclasse com desbalanceamento moderado** (ver 2.2). A acurácia sozinha esconde o desempenho nas classes pequenas, então:

| Métrica | Papel |
|---|---|
| **F1-macro** (validação) | **Métrica principal e regra de seleção** (early stopping, melhor época, comparação das três condições). Dá o mesmo peso a todas as classes |
| Acurácia balanceada (= recall médio por classe) | Mostra se alguma classe está sendo "esquecida" |
| Acurácia | Leitura intuitiva, comparável com outros trabalhos |
| Recall e precisão por classe + matriz de confusão | Mostram **quais** classes se confundem |

**Erros mais prejudiciais.** Pensando numa finalidade de catalogação automática de fotos de mergulho ou monitoramento de biodiversidade:
- Confundir grupos **visualmente parecidos e biologicamente distintos** (caranguejo × lagosta × camarão; foca × lontra; tubarão × golfinho × baleia) distorce contagens de espécies. Esses são os erros a vigiar na matriz de confusão.
- Uma classe com **recall baixo** significa animais daquele tipo sistematicamente não catalogados. O F1-macro e a acurácia balanceada caem quando isso acontece, mesmo que a acurácia geral continue alta.

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "bal_acc": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

## 3. Inception-v4 com base congelada (etapa 3)

**Implementação:** PyTorch + `timm` (`inception_v4.tf_in1k`, pesos ImageNet-1k convertidos do TF-Slim oficial).
O Keras não tem Inception-v4 pronta (só a v3 e a Inception-ResNet-v2), por isso PyTorch.

**Adaptação:** o classificador original (`last_linear`: 1536 → 1000 classes do ImageNet) é substituído por **dropout (0,3) + linear 1536 → nº de classes**. Toda a base convolucional fica congelada; só esse classificador novo é treinado.

**Detalhe importante:** as camadas de **BatchNorm da base congelada ficam em modo de avaliação** durante o treino. Se ficassem em modo de treino, as médias e variâncias móveis seriam atualizadas com as estatísticas do nosso dataset, alterando a base "congelada" sem nenhum gradiente. A função `set_train_mode` cuida disso e serve também para a parte 3: no fine-tuning, os BatchNorm das camadas descongeladas voltam a treinar.

In [ ]:
def build_model(num_classes=None, pretrained=PRETRAINED):
    return timm.create_model(MODEL_NAME, pretrained=pretrained, num_classes=num_classes or NUM_CLASSES,
                             drop_rate=DROP_RATE)

def freeze_base(model):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.get_classifier().parameters():
        p.requires_grad = True

def set_train_mode(model):
    # Treino normal, mas BatchNorm sem parâmetros treináveis fica em eval (estatísticas congeladas).
    model.train()
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters()):
            m.eval()

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

seed_everything()
model = build_model().to(DEVICE)
freeze_base(model)

total, trainable = count_params(model)
with torch.no_grad():
    out = model.eval()(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
    feats = model.forward_head(model.forward_features(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)),
                               pre_logits=True)
print(f"Entrada: (N, 3, {IMG_SIZE}, {IMG_SIZE})  ->  vetor de features: {tuple(feats.shape)}  ->  saída: {tuple(out.shape)}")
print("Classificador novo:", model.get_classifier())
print(f"Parâmetros: total {total:,} | treináveis {trainable:,} ({trainable / total:.3%}) | congelados {total - trainable:,}")

In [ ]:
# Estrutura em alto nível (stem + blocos Inception-A/B/C + reduções), para relacionar com o diagrama do artigo
blocos = pd.DataFrame([(i, type(m).__name__, sum(p.numel() for p in m.parameters()))
                       for i, m in enumerate(model.features)], columns=["índice", "bloco", "parâmetros"])
blocos

### 3.1 Treinamento

| Hiperparâmetro | Valor |
|---|---|
| Otimizador | AdamW (só parâmetros treináveis), weight decay 1e-4 |
| Learning rate | 1e-3, reduzida pela metade se o F1-macro da validação estagnar por 2 épocas (`ReduceLROnPlateau`) |
| Batch size | 32 |
| Épocas | até 20, **early stopping** com paciência 4 no F1-macro da validação |
| Função de perda | Entropia cruzada |
| Seleção | Pesos da **época de maior F1-macro na validação** são restaurados e salvos |
| Precisão | Mista (AMP) na GPU |

A função `fit` abaixo é a mesma que a parte 3 deve usar no fine-tuning (mudando só o que for parte da estratégia), para manter as condições comparáveis.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    set_train_mode(model) if training else model.eval()
    losses, ys, preds, probs = [], [], [], []
    use_amp = DEVICE.type == "cuda"
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            losses.append(loss.item() * len(y))
            p = logits.float().softmax(1)
            ys.append(y.cpu()); preds.append(p.argmax(1).cpu()); probs.append(p.detach().cpu())
    y, yhat = torch.cat(ys).numpy(), torch.cat(preds).numpy()
    return sum(losses) / len(y), y, yhat, torch.cat(probs).numpy()

def fit(model, train_loader, val_loader, optimizer, scheduler=None, max_epochs=MAX_EPOCHS,
        patience=PATIENCE, label="modelo"):
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best, best_state, best_epoch, wait, hist = -1.0, None, 0, 0, []
    t0 = time.time()
    for epoch in range(1, max_epochs + 1):
        te = time.time()
        tr_loss, ytr, ptr, _ = run_epoch(model, train_loader, criterion, optimizer, scaler)
        va_loss, yva, pva, _ = run_epoch(model, val_loader, criterion)
        mtr, mva = compute_metrics(ytr, ptr), compute_metrics(yva, pva)
        row = {"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss,
               **{f"train_{k}": v for k, v in mtr.items()}, **{f"val_{k}": v for k, v in mva.items()},
               "lr": optimizer.param_groups[0]["lr"], "seg": time.time() - te}
        hist.append(row)
        if scheduler is not None:
            scheduler.step(row[MONITOR])
        melhorou = row[MONITOR] > best + 1e-4
        if melhorou:
            best, best_epoch, wait = row[MONITOR], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
        print(f"[{label}] ép {epoch:02d} | loss tr {tr_loss:.4f} va {va_loss:.4f} | F1 tr {mtr['f1_macro']:.4f} "
              f"va {mva['f1_macro']:.4f} | acc va {mva['acc']:.4f} | lr {row['lr']:.1e} | {row['seg']:.0f}s"
              + ("  *" if melhorou else ""))
        if wait >= patience:
            print(f"Early stopping: {patience} épocas sem melhora.")
            break
    model.load_state_dict(best_state)
    hist = pd.DataFrame(hist)
    info = {"best_epoch": best_epoch, "epochs_run": len(hist), "best_" + MONITOR: best,
            "train_time_s": time.time() - t0}
    print(f"Melhor época: {best_epoch} ({MONITOR} = {best:.4f}) | tempo total {info['train_time_s'] / 60:.1f} min")
    return hist, info

In [ ]:
seed_everything()
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

hist_base, info_base = fit(model, train_loader, val_loader, optimizer, scheduler, label="base congelada")
hist_base.to_csv(ARTIFACTS / "history_base_congelada.csv", index=False)

### 3.2 Curvas de aprendizagem

In [ ]:
def plot_history(hist, title, best_epoch=None):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(hist.epoch, hist.train_loss, "o-", label="treino")
    axes[0].plot(hist.epoch, hist.val_loss, "o-", label="validação")
    axes[0].set_title("Loss (entropia cruzada)"); axes[0].set_xlabel("época")
    axes[1].plot(hist.epoch, hist.train_f1_macro, "o-", label="treino")
    axes[1].plot(hist.epoch, hist.val_f1_macro, "o-", label="validação")
    axes[1].set_title("F1-macro"); axes[1].set_xlabel("época")
    for ax in axes:
        if best_epoch:
            ax.axvline(best_epoch, c="gray", ls=":", label=f"melhor época ({best_epoch})")
        ax.legend()
    fig.suptitle(title); plt.tight_layout(); plt.show()

plot_history(hist_base, "Inception-v4 — base congelada", info_base["best_epoch"])
hist_base.round(4)

**Como ler as curvas** (escrever a análise depois de rodar com os resultados reais):

- Com a base congelada, a capacidade treinável é só o classificador linear, então **overfitting forte é pouco provável**. É comum o F1 de *treino* ficar **abaixo** do de validação, porque o treino é medido com aumento de dados e dropout ativos e a validação não.
- Se treino e validação estabilizam juntos em um patamar abaixo do desejado, é sinal de **underfitting/limite das features do ImageNet**. É exatamente o que o fine-tuning (parte 3) tenta superar, adaptando as camadas finais à aparência das imagens marinhas.
- Se a loss de validação sobe enquanto a de treino cai, há início de overfitting, e o early stopping já guardou a época anterior.

### 3.3 Resultados na validação

In [ ]:
criterion = nn.CrossEntropyLoss()
val_loss, yv, pv, probv = run_epoch(model, val_loader, criterion)
m_val = compute_metrics(yv, pv)
print("Validação (melhor época):", {k: round(v, 4) for k, v in m_val.items()}, f"| loss {val_loss:.4f}")

rep = pd.DataFrame(classification_report(yv, pv, labels=range(NUM_CLASSES), target_names=classes,
                                         output_dict=True, zero_division=0)).T
rep_cls = rep.iloc[:NUM_CLASSES].sort_values("f1-score")
rep_cls.round(3)

In [ ]:
cm = confusion_matrix(yv, pv, labels=range(NUM_CLASSES))
cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(0.55 * NUM_CLASSES + 3, 0.5 * NUM_CLASSES + 2))
sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes,
            cbar_kws={"label": "proporção da classe real"}, ax=ax, annot_kws={"size": 7})
ax.set_xlabel("classe prevista"); ax.set_ylabel("classe real")
ax.set_title("Matriz de confusão — validação, base congelada (cor = % da linha, número = contagem)")
plt.tight_layout(); plt.show()

conf = pd.DataFrame([(classes[i], classes[j], cm[i, j], cm_norm[i, j])
                     for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0],
                    columns=["real", "prevista", "n", "% da classe real"])
print("Principais confusões:")
conf.sort_values("n", ascending=False).head(10).round(3)

### 3.4 Análise visual preliminar (acertos e erros)

Cada imagem mostra **classe real**, **classe prevista** e **score** (probabilidade da classe prevista). As previsões de validação ficam salvas em `artifacts/val_preds_base_congelada.csv` para que a parte 3 acompanhe **as mesmas imagens** nos três modelos (erros corrigidos, persistentes e novos).

In [ ]:
val_df = split[split.split == "val"].reset_index(drop=True).copy()
val_df["pred"] = [classes[i] for i in pv]
val_df["score"] = probv.max(1)
val_df["p_real"] = probv[np.arange(len(yv)), yv]
val_df["acertou"] = val_df.pred == val_df.label
for i, c in enumerate(classes):
    val_df[f"prob_{c}"] = probv[:, i]
val_df.to_csv(ARTIFACTS / "val_preds_base_congelada.csv", index=False)

def titles(d):
    return [f"real: {r.label}\nprev: {r.pred} ({r.score:.2f})" for r in d.itertuples()]

erros = val_df[~val_df.acertou].sort_values("score", ascending=False)
print(f"Erros na validação: {len(erros)} de {len(val_df)}")
top = erros.head(12)
show_grid(top.path.tolist(), titles(top), suptitle="Erros mais 'confiantes' (score alto na classe errada)")

acertos = val_df[val_df.acertou]
inc = acertos.sort_values("score").head(6)
seg = acertos.sort_values("score", ascending=False).head(6)
show_grid(inc.path.tolist() + seg.path.tolist(), titles(inc) + titles(seg),
          suptitle="Acertos: 6 mais incertos (1ª linha) e 6 mais seguros (2ª linha)")

**Hipóteses a observar** (completar com o que aparecer nas imagens): erros entre animais de formato parecido; imagens com vários animais de classes diferentes na mesma foto; animal pequeno com o fundo dominando (o modelo pode estar usando o fundo, o que será verificado com Grad-CAM na parte 3); desenhos, ilustrações ou fotos fora d'água; possíveis rótulos errados no próprio dataset.

### 3.5 Salvar o modelo de referência

O checkpoint guarda os pesos **e** tudo que é preciso para reutilizá-lo: nome da arquitetura, lista de classes na ordem dos índices e configuração de pré-processamento. As estratégias A e B carregam **cópias deste mesmo arquivo**.

> O arquivo tem cerca de 160 MB, acima do limite do GitHub. Por isso fica fora do git (`.gitignore`): subir no Drive da equipe e colocar o link no README.

In [ ]:
ckpt_path = ARTIFACTS / "modelo_base_congelada.pt"
torch.save({
    "model_name": MODEL_NAME,
    "num_classes": NUM_CLASSES,
    "drop_rate": DROP_RATE,
    "classes": classes,
    "class_to_idx": class_to_idx,
    "data_config": {"img_size": IMG_SIZE, "resize": RESIZE, "mean": MEAN, "std": STD, "interpolation": "bicubic"},
    "state_dict": model.state_dict(),
    "condition": "base_congelada",
    "train_info": info_base,
    "hparams": {"lr": LR, "weight_decay": WEIGHT_DECAY, "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS,
                "patience": PATIENCE, "monitor": MONITOR, "optimizer": "AdamW", "scheduler": "ReduceLROnPlateau(0.5, 2)",
                "seed": SEED},
}, ckpt_path)

resumo = {
    "condicao": "base congelada",
    "params_total": total, "params_treinaveis": trainable,
    **info_base,
    **{f"val_{k}": v for k, v in m_val.items()}, "val_loss": val_loss,
    "n_train": int((split.split == "train").sum()), "n_val": len(val_df), "n_test": int((split.split == "test").sum()),
}
json.dump(resumo, open(ARTIFACTS / "resumo_base_congelada.json", "w"), indent=2, ensure_ascii=False, default=float)
print(f"Salvo: {ckpt_path} ({ckpt_path.stat().st_size / 1e6:.0f} MB)")
pd.Series(resumo)

### 3.6 Conferência: carregar o checkpoint do zero

É assim que a parte 3 deve iniciar cada estratégia de fine-tuning (cada uma com sua própria cópia). Também confirma que o arquivo salvo reproduz exatamente as previsões de validação.

In [ ]:
def load_checkpoint(path, device=DEVICE):
    ck = torch.load(path, map_location=device, weights_only=False)
    m = timm.create_model(ck["model_name"], pretrained=False, num_classes=ck["num_classes"], drop_rate=ck["drop_rate"])
    m.load_state_dict(ck["state_dict"])
    return m.to(device), ck

model_copy, ck = load_checkpoint(ckpt_path)
_, _, pv2, _ = run_epoch(model_copy, val_loader, criterion)
assert (pv2 == pv).all(), "O checkpoint carregado não reproduz as previsões!"
print("Checkpoint OK: mesmas previsões na validação. Classes:", ck["classes"][:5], "...")
del model_copy

---
**O conjunto de teste não foi usado neste notebook.** Ele só será avaliado na etapa 7, nos três modelos ao mesmo tempo, depois que o modelo recomendado for escolhido pela validação.